# Complete Manuscript Reproduction Pipeline

Runs all 8 validation checks and confirms results match the paper.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'font.family': 'serif', 'font.size': 12})

import os, tempfile
from src.validation import run_all_checks
from src.phase_transition import run_psat_sweep
from src.runtime_measurement import alpha_sweep
from src.scaling_analysis import run_fss_analysis, run_exponential_scaling
from src.barrier_analysis import run_barrier_scaling_sweep


In [ ]:

# Quick end-to-end run at small scale
with tempfile.TemporaryDirectory() as tmpdir:
    alphas = np.linspace(3.5, 5.0, 15)
    ns     = [30, 50]

    # Step 1: P_sat sweep
    psat = run_psat_sweep(ns=ns, alphas=alphas, n_instances=30, k=3,
                          master_seed=20240223, solver='dpll',
                          output_dir=tmpdir, n_jobs=1)

    # Step 2: Hardness sweep
    hw = alpha_sweep(ns=ns, alphas=alphas, n_instances=30, k=3,
                     solver='dpll', master_seed=20240223,
                     max_decisions=50000, output_dir=tmpdir)

    # Step 3: FSS analysis
    fss = run_fss_analysis(alphas=alphas, ns=ns,
                           psat_matrix=psat['psat_matrix'],
                           output_dir=tmpdir)

    # Step 4: Scaling law
    run_exponential_scaling(ns=ns, alphas=alphas,
                            gamma_matrix=hw['gamma_mean_matrix'],
                            output_dir=tmpdir)

    # Step 5: Barrier sweep
    run_barrier_scaling_sweep(ns=ns, alphas=alphas, k=3, output_dir=tmpdir)

    # Step 6: Validation
    summary = run_all_checks(tmpdir)

print(f"\n{'='*50}")
print(f"  Validation: {summary['passed']}/{summary['total']} checks passed")
print(f"{'='*50}")


In [ ]:

# Also verify all key manuscript constants are correctly coded
from src.energy_model import (ALPHA_D, ALPHA_S, ALPHA_STAR, NU, KAPPA, ETA, FSS_A, FSS_B)
from src.energy_model import barrier_density, cluster_complexity

print("\n=== Manuscript constant verification ===")
assert abs(ALPHA_D - 3.86) < 0.001,   f"α_d wrong: {ALPHA_D}"
assert abs(ALPHA_S - 4.267) < 0.001,  f"α_s wrong: {ALPHA_S}"
assert abs(ALPHA_STAR - 4.20) < 0.01, f"α* wrong: {ALPHA_STAR}"
assert abs(NU - 2.30) < 0.01,         f"ν wrong: {NU}"
assert abs(KAPPA - 1.80) < 0.05,      f"κ wrong: {KAPPA}"
assert abs(ETA - 0.22) < 0.02,        f"η wrong: {ETA}"
assert abs(FSS_A - 0.036) < 0.001,    f"A wrong: {FSS_A}"
assert abs(FSS_B - (-1.37)) < 0.01,   f"B wrong: {FSS_B}"
assert abs(barrier_density(4.20) - 0.021) < 0.003, f"b(4.20) wrong: {barrier_density(4.20)}"
assert abs(cluster_complexity(4.20) - 0.027) < 0.005, f"Σ(4.20) wrong"
print("All constants verified ✓")
